# AWS Credit card fraud detection 

In this solution, we'll create the groundwork for a credit card fraud detection system using SageMaker. We'll kick off by developing an anomaly detection model, then build two supervised XGBoost models. Given that fraud detection works with severely unbalanced datasets, our first XGBoost model will handle this by adjusting data weights, while the second will apply SMOTE to generate more examples of the uncommon fraud instances.

We'll also demonstrate how to interact with a REST API to mimic a production environment, leveraging AWS Lambda to trigger both the anomaly detection and XGBoost models.

Ready to execute everything in one go? Simply choose Run → Run All in Studio, or Cell → Run All if you're using a SageMaker Notebook Instance.

In [ ]:
import os
import zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import IsolationForest
import xgboost as xgb
import boto3
import joblib
from dotenv import load_dotenv
load_dotenv()

In [ ]:
import sys
sys.path.insert(0, '.')

### Set up environment

Let's set up environment

In [ ]:
# Configuration des variables d'environnement
# Sur SageMaker, les credentials sont fournis automatiquement par l'IAM execution role
aws_region = os.environ.get('AWS_REGION', boto3.Session().region_name)
s3_bucket = os.getenv("SPARK_DATA_BUCKET", "fraud-detection-data-bkt-eu-west-1")
s3_prefix = os.getenv("SOLUTION_NAME", "fraud-detection")

print(f"aws_region: {aws_region}")
print(f"s3_bucket: {s3_bucket}")
print(f"s3_prefix: {s3_prefix}")

In [ ]:
DATASET_PATH = 'dataset'
os.makedirs(DATASET_PATH, exist_ok=True)
# os.makedirs(CHECKPOINTS_PATH, exist_ok=True)

In [ ]:
# Initialisation du client S3
s3_client = boto3.client('s3', region_name=aws_region)

In [ ]:
# Download creditcard.csv from S3
s3_key = "dataset/creditcard.csv"
local_csv_path = f"{DATASET_PATH}/creditcard.csv"

print("Téléchargement en cours...")
s3_client.download_file(s3_bucket, s3_key, local_csv_path)
print(f"Téléchargement terminé : {local_csv_path}")

In [ ]:
print(f"Dataset prêt : {local_csv_path}")

## Sagemaker Setup

In [ ]:
import sagemaker

# sagemaker_iam_role = os.getenv("SAGEMAKER_IAM_ROLE")
sagemaker_iam_role = sagemaker.get_execution_role()
sagemaker_session = sagemaker.Session()
default_bucket = sagemaker_session.default_bucket()

## Investigate and process the data

Let's start by reading in the credit card fraud data set.

In [ ]:
data = pd.read_csv(f"{DATASET_PATH}/creditcard.csv", delimiter=',')
data.head()

Let's take a peek at our data (we only show a subset of the columns in the table):

In [ ]:
print(data.columns)
data[['Time', 'V1', 'V2', 'V27', 'V28', 'Amount', 'Class']].describe()

The dataset contains only numerical values since the original features were transformed using PCA for privacy protection. This leaves us with 28 new features (V1–V28), along with two untransformed ones: Amount (the transaction value) and Time (seconds elapsed since the first transaction in the dataset).

Our target column indicates whether each transaction is fraudulent. The data is extremely skewed: among 284,807 transactions, just 492 are fraudulent — roughly 0.17% of all cases.

In [ ]:
nonfrauds, frauds = data.groupby('Class').size()
print('Number of frauds: ', frauds)
print('Number of non-frauds: ', nonfrauds)
print('Percentage of fradulent data:', 100.*frauds/(frauds + nonfrauds))

We know the V_i columns have already been standardized to zero mean and unit variance as part of the PCA transformation.

In [ ]:
feature_columns = data.columns[:-1]
label_column = data.columns[-1]

features = data[feature_columns].values.astype('float32')
labels = (data[label_column].values).astype('float32')

Next, we will prepare our data for loading and training.

## Training

Let's begin by dividing our data into training and test sets to evaluate model performance later. We need to do this split first, before applying any techniques to handle the class imbalance - otherwise we might accidentally leak information from the test set into our training data.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    features, labels, test_size=0.1, random_state=42
)

> Note: If you're working with your own dataset that has categorical features with text values, you'll need to convert them to numbers first. One-hot encoding is a popular approach - you can use scikit-learn's [OneHotEncoder](https://scikit-learn.org/stable/modules/preprocessing.html#preprocessing-categorical-features) for this. This conversion is essential since XGBoost only handles numerical data.

## Supervised Learning

Once we have sufficient labeled training data, we can move to supervised learning, where the model discovers patterns connecting the features to the target class.

We'll use XGBoost for this task. It's an excellent choice because it has a proven track record in real-world applications, scales well with large datasets, and manages missing values without needing extensive preprocessing.

### Prepare Data and Upload to S3

First we copy the data to an in-memory buffer.

In [ ]:
import io
import sklearn
from sklearn.datasets import dump_svmlight_file


In [ ]:
buf = io.BytesIO()

sklearn.datasets.dump_svmlight_file(X_train, y_train, buf)
buf.seek(0)

Now we upload the data to S3 using boto3.

In [ ]:
import os

# data_location = 's3://{}/{}/train'.format(default_bucket, s3_prefix)
base_job_name = "{}-xgb".format(s3_prefix)
s3_output_path = 's3://{}/output/default-output'.format(default_bucket)
data_key = 'fraud-dataset'

# create train data s3 dir and store x_train into it.
boto3.resource('s3', region_name=aws_region).Bucket(default_bucket).Object(os.path.join('data', 'train', data_key)).upload_fileobj(buf)
s3_train_data_path = 's3://{}/data/train/{}'.format(default_bucket, data_key)

In [ ]:
print('Training artifacts will be uploaded to: {}'.format(s3_output_path))
print('Sagemaker IAM role: {}'.format(sagemaker_iam_role))
print('Sagemaker S3 Default bucket: {}'.format(default_bucket))
print('Training artifacts will be uploaded to: {}'.format(s3_output_path))
print('Uploaded training data location: {}'.format(s3_train_data_path))

Now we can train our model using SageMaker's built-in XGBoost algorithm. We'll use a utility function to get the algorithm's URI. You can find the complete list of built-in algorithms here: https://docs.aws.amazon.com/sagemaker/latest/dg/algos.html

In [ ]:
import sagemaker

# Get the XGBoost image URI
xgboost_image_uri = sagemaker.image_uris.retrieve(
    framework="xgboost",
    region=boto3.Session().region_name,
    version="0.90-2",
    py_version="py3",
)

SageMaker handles training through Estimators. You specify the model type, its settings, and hyperparameters, then tell the Estimator where to find your training data in S3.

For our fraud detection problem, there's one crucial hyperparameter: scale_pos_weight. This balances the importance of positive (fraud) versus negative (non-fraud) examples during training. With our heavily skewed dataset, this parameter is essential — without it, the model would focus almost entirely on the abundant non-fraud cases and miss the rare fraud patterns.

In [ ]:
from math import sqrt

# Because the data set is so highly skewed, we set the scale position weight conservatively,
# as sqrt(num_nonfraud/num_fraud).
# Other recommendations for the scale_pos_weight are setting it to (num_nonfraud/num_fraud).
scale_pos_weight = sqrt(np.count_nonzero(y_train == 0)/np.count_nonzero(y_train))
hyperparams = {
    "max_depth":5,
    "subsample":0.8,
    "num_round":100,
    "eta":0.2,
    "gamma":4,
    "min_child_weight":6,
    "silent":0,
    "objective":'binary:logistic',
    "eval_metric":'auc',
    "scale_pos_weight": scale_pos_weight
}

Let's break down the hyperparameters we're using.

The most critical one for imbalanced data is scale_pos_weight. This controls how much extra importance the model gives to fraud cases versus legitimate ones. While you could use the full ratio (num_nonfraud / num_fraud), our dataset is so extremely skewed that we'll use the square root instead:
sqrt(num_nonfraud / num_fraud).

For our data, that's sqrt(284,807 / 492) ≈ 24. So each fraud case gets treated as roughly 24 times more important than a non-fraud case during training.

Here are the other key hyperparameters:

- max_depth – How deep each tree can grow. At depth 5, trees can have up to 32 leaves. Trees grow exponentially (2^max_depth leaves), so deeper trees overfit quickly. Depth 10 would mean 1024 leaves!

- subsample – What fraction of data trains each tree. At 0.8, each tree sees a random 80% sample, which helps prevent overfitting.

- num_round – How many trees to add to the ensemble. We're using 100 boosting rounds.

- eta – The learning rate that shrinks each new tree's contribution, helping reduce overfitting.

- gamma – Minimum improvement needed before splitting a leaf. Higher values mean the model only makes splits that really help, preventing overfitting.

- min_child_weight – Like gamma, this sets the bar for how much gain is required before making a split. Higher values make the model more cautious.

- objective – We're using logistic loss since this is binary classification.

- eval_metric – The evaluation metric matters a lot with imbalanced data. We'll use AUC, which handles this situation well.

In [ ]:
from sagemaker.estimator import Estimator

xgb_clf = Estimator(
    sagemaker_session=sagemaker_session,
    role=sagemaker_iam_role,
    image_uri=xgboost_image_uri,
    instance_count=1,
    instance_type='ml.m4.xlarge',
    hyperparameters=hyperparams,
    output_path=s3_output_path,
    base_job_name="{}-xgb".format(s3_prefix)
)

We can now fit our supervised training model, the call to fit below should take around 5 minutes to complete.

In [ ]:
xgb_clf.fit({'train': s3_train_data_path})

In [ ]:
# Get the path of the trained model artifact in S3
s3_model_file_path = xgb_clf.latest_training_job.describe()['ModelArtifacts']['S3ModelArtifacts']
print(s3_model_file_path)

### Analyse model

In [ ]:
# import tarfile, xgboost

In [ ]:
# s3 = boto3.client('s3')

In [ ]:
# # Download the trained model artifact to the local environment
# s3.download_file(default_bucket, s3_model_file_path.split(default_bucket + '/')[1], 'model.tar.gz')

In [ ]:
# # Extract the model artifact
# # with tarfile.open('model.tar.gz') as tar: tar.extractall()

# with tarfile.open('model.tar.gz') as tar:
#     tar.extractall(filter='data')

In [ ]:
# # Load the model and set its feature names
# xgb_model = xgboost.Booster()
# xgb_model.load_model('xgboost-model')
# xgb_model.feature_names = list(train.columns[1:])

In [ ]:
# # Plot the feature importance of the trained model
# fig, ax = plt.subplots()
# xgboost.plot_importance(xgb_model, ax=ax)
# plt.show()

### Host Classifier

Time to deploy our estimator to an endpoint. You'll see those `-` progress indicators again, and deployment should wrap up in roughly 10 minutes.

In [ ]:
# # --- Initialisation ---
# sm_client = boto3.client("sagemaker")

# # --- Paramètres ---
# s3_prefix = "fraud-detection"
# endpoint_name = f"{s3_prefix}-xgb-endpoint"
# model_name = f"{s3_prefix}-xgb"

# # --- Suppression si l'endpoint existe déjà ---
# def delete_endpoint_and_config(endpoint_name):

#     try:
#         # Supprimer l'endpoint
#         print(f"Deleting endpoint: {endpoint_name}")
#         sm_client.delete_endpoint(EndpointName=endpoint_name)

#         # Attendre la suppression effective
#         waiter = sm_client.get_waiter('endpoint_deleted')
#         waiter.wait(EndpointName=endpoint_name)

#     except sm_client.exceptions.ClientError as e:
#         if "Could not find endpoint" in str(e):
#             print(f"Endpoint {endpoint_name} does not exist. Skipping delete.")
#         else:
#             raise e

#     try:
#         # Supprimer l'endpoint config
#         print(f"Deleting endpoint config: {endpoint_name}")
#         sm_client.delete_endpoint_config(EndpointConfigName=endpoint_name)
#     except sm_client.exceptions.ClientError as e:
#         if "Could not find endpoint configuration" in str(e):
#             print(f"Endpoint config {endpoint_name} does not exist. Skipping delete.")
#         else:
#             raise e

# # --- Cleanup ---
# delete_endpoint_and_config(endpoint_name)

In [ ]:
from sagemaker.serializers import CSVSerializer

dpl_xgb_clf = xgb_clf.deploy(
    initial_instance_count=1,
    model_name="{}-xgb".format(s3_prefix),
    endpoint_name="{}-xgb-endpoint".format(s3_prefix),
    instance_type='ml.m4.xlarge',
    serializer=CSVSerializer(),
    deserializer=None
)

### Evaluation

In [ ]:
from sklearn.metrics import balanced_accuracy_score, cohen_kappa_score

Now that we've trained the model, we can use it to make predictions on our test data.

In [ ]:
# Because we have a large test set, we call predict on smaller batches
def predict(classifier, data, rows=500):
    split_array = np.array_split(data, int(data.shape[0] / float(rows) + 1))
    predictions = ''
    for array in split_array:
        predictions = ','.join([predictions, classifier.predict(array, initial_args={'ContentType': 'text/csv'}).decode('utf-8')])

    return np.fromstring(predictions[1:], sep=',')

In [ ]:
raw_preds = predict(dpl_xgb_clf, X_test)

Let's check how our model did using some scikit-learn metrics. With an imbalanced dataset like ours, we need metrics that consider how often each class appears in the data.

Two excellent choices are the [balanced accuracy score](https://scikit-learn.org/stable/modules/model_evaluation.html#balanced-accuracy-score) and [Cohen's Kappa](https://scikit-learn.org/stable/modules/model_evaluation.html#cohen-s-kappa).

In [ ]:
# scikit-learn expects 0/1 predictions, so we threshold our raw predictions
y_preds = np.where(raw_preds > 0.5, 1, 0)
print("Balanced accuracy = {}".format(balanced_accuracy_score(y_test, y_preds)))
print("Cohen's Kappa = {}".format(cohen_kappa_score(y_test, y_preds)))

These results look fantastic! Our model is doing really well on both measures. Cohen's Kappa scores over 0.8 are typically seen as outstanding performance, so we're definitely on the right track.

These overall scores are great, but let's dig deeper into how our model handles each class separately. A confusion matrix along with precision, recall, and f1-scores for each class will show us exactly where the model excels and where it struggles.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report

def plot_confusion_matrix(y_true, y_predicted):

    cm  = confusion_matrix(y_true, y_predicted)
    # Get the per-class normalized value for each cell
    cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

    # We color each cell according to its normalized value, annotate with exact counts.
    ax = sns.heatmap(cm_norm, annot=cm, fmt="d")
    ax.set(xticklabels=["non-fraud", "fraud"], yticklabels=["non-fraud", "fraud"])
    ax.set_ylim([0,2])
    plt.title('Confusion Matrix')
    plt.ylabel('Real Classes')
    plt.xlabel('Predicted Classes')
    plt.show()

In [ ]:
plot_confusion_matrix(y_test, y_preds)

In [ ]:
print(
    classification_report(
        y_test, y_preds, target_names=['non-fraud', 'fraud']
    )
)

In [ ]:
# # --- Initialisation ---
# sm_client = boto3.client("sagemaker")

# # --- Paramètres ---
# s3_prefix = "fraud-detection"
# endpoint_name = f"{s3_prefix}-xgb-endpoint"
# model_name = f"{s3_prefix}-xgb"

# # --- Suppression si l'endpoint existe déjà ---
# def delete_endpoint_and_config(endpoint_name):

#     try:
#         # Supprimer l'endpoint
#         print(f"Deleting endpoint: {endpoint_name}")
#         sm_client.delete_endpoint(EndpointName=endpoint_name)

#         # Attendre la suppression effective
#         waiter = sm_client.get_waiter('endpoint_deleted')
#         waiter.wait(EndpointName=endpoint_name)

#     except sm_client.exceptions.ClientError as e:
#         if "Could not find endpoint" in str(e):
#             print(f"Endpoint {endpoint_name} does not exist. Skipping delete.")
#         else:
#             raise e

#     try:
#         # Supprimer l'endpoint config
#         print(f"Deleting endpoint config: {endpoint_name}")
#         sm_client.delete_endpoint_config(EndpointConfigName=endpoint_name)
#     except sm_client.exceptions.ClientError as e:
#         if "Could not find endpoint configuration" in str(e):
#             print(f"Endpoint config {endpoint_name} does not exist. Skipping delete.")
#         else:
#             raise e

# # --- Cleanup ---
# delete_endpoint_and_config(endpoint_name)

In [ ]:
# Uncomment to clean up endpoints
dpl_xgb_clf.delete_model()
dpl_xgb_clf.delete_endpoint()
sagemaker_client = boto3.client('sagemaker', region_name=aws_region)
waiter = sagemaker_client.get_waiter('endpoint_deleted')
waiter.wait(EndpointName="{}-xgb-endpoint".format(s3_prefix))

### SMOTE for Data augmentation

With our baseline XGBoost model in hand, let's try to improve performance using sampling methods built for imbalanced data.

We'll work with the [imbalanced-learn](https://imbalanced-learn.readthedocs.io/en/stable/index.html) library, which integrates seamlessly with scikit-learn. It's pre-installed here, but for other environments, just run `pip install --upgrade imbalanced-learn` in your conda setup.

We'll apply [Synthetic Minority Over-sampling](https://arxiv.org/abs/1106.1813) (SMOTE), which generates new minority class examples by creating synthetic points between existing ones.

In [ ]:
import sys
!{sys.executable} -m pip install imblearn

from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=42)
X_smote, y_smote = smote.fit_resample(X_train, y_train)

We can see that SMOTE has now balanced the two classes:

In [ ]:
from collections import Counter
print(sorted(Counter(y_smote).items()))

We went pretty overboard with the oversampling here - jumping from ~0.17% all the way to 50% minority class! You could take a gentler approach, like creating one minority sample for every `sqrt(non_fraud/fraud)` majority samples, or try other sophisticated resampling methods. Take a look at this [oversampling techniques comparison](https://imbalanced-learn.readthedocs.io/en/stable/auto_examples/over-sampling/plot_comparison_over_sampling.html#sphx-glr-auto-examples-over-sampling-plot-comparison-over-sampling-py) from imbalanced-learn to explore what's available.

In our case we'll use the SMOTE dataset we just created and upload it to S3 for training.

In [ ]:
smote_buf = io.BytesIO()

# Dump the SMOTE data into a buffer
sklearn.datasets.dump_svmlight_file(X_smote, y_smote, smote_buf)
smote_buf.seek(0)

# Upload from the buffer to S3

data_prefix = 'data'
data_key = 'fraud-dataset-smote'
boto3.resource('s3', region_name=aws_region).Bucket(default_bucket).Object(os.path.join(data_prefix,'train', data_key)).upload_fileobj(smote_buf)


s3_smote_train_data_path = 's3://{}/data/train/{}'.format(default_bucket, data_key)
print('Uploaded training data location: {}'.format(s3_smote_train_data_path))

s3_smote_output_path = 's3://{}/output/smote-output'.format(default_bucket)
print('Training artifacts will be uploaded to: {}'.format(s3_smote_output_path))

In [ ]:
# No need to scale weights after SMOTE resampling, so we remove that parameter
hyperparams.pop("scale_pos_weight", None)
smote_xgb_clf = sagemaker.estimator.Estimator(
    xgboost_image_uri,
    role=sagemaker_iam_role,
    hyperparameters=hyperparams,
    instance_count=1,
    instance_type='ml.m4.xlarge',
    output_path=s3_smote_output_path,
    sagemaker_session=sagemaker_session,
    base_job_name="{}-xgb-smote".format(s3_prefix)
)

We are now ready to fit the model, which should take around 5 minutes to complete.

In [ ]:
smote_xgb_clf.fit({'train': s3_smote_train_data_path})

After fitting the model we can check its performance to compare it against the base XGBoost model. The deployment will take around 10 minutes.

In [ ]:
from sagemaker.serializers import CSVSerializer
from sagemaker.deserializers import CSVDeserializer

dpl_smote_xgb_clf = smote_xgb_clf.deploy(
    initial_instance_count=1,
    model_name="{}-xgb-smote".format(s3_prefix),
    endpoint_name="{}-xgb-smote-endpoint".format(s3_prefix),
    instance_type='ml.m4.xlarge'
)

# Specify input and output formats.
dpl_smote_xgb_clf.content_type = 'text/csv'
csv_serializer = CSVSerializer()
dpl_smote_xgb_clf.serializer = csv_serializer

# Set the deserializer to handle the response from the inference endpoint
# csv_deserializer = CSVDeserializer()
# dpl_smote_xgb_clf.deserializer = csv_deserializer

# Evaluation SMOTE

In [ ]:
# Because we have a large test set, we call predict on smaller batches
def predict(classifier, data, rows=500):
    split_array = np.array_split(data, int(data.shape[0] / float(rows) + 1))
    predictions = ''
    for array in split_array:
        predictions = ','.join([predictions, classifier.predict(array, initial_args={'ContentType': 'text/csv'}).decode('utf-8')])

    return np.fromstring(predictions[1:], sep=',')

In [ ]:
smote_raw_preds = predict(dpl_smote_xgb_clf, X_test)
smote_preds = np.where(smote_raw_preds > 0.5, 1, 0)

In [ ]:
print("Balanced accuracy = {}".format(balanced_accuracy_score(y_test, smote_preds)))
print("Cohen's Kappa = {}".format(cohen_kappa_score(y_test, smote_preds)))

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

In [ ]:
def plot_confusion_matrix(y_true, y_predicted):

    cm  = confusion_matrix(y_true, y_predicted)
    # Get the per-class normalized value for each cell
    cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

    # We color each cell according to its normalized value, annotate with exact counts.
    ax = sns.heatmap(cm_norm, annot=cm, fmt="d")
    ax.set(xticklabels=["non-fraud", "fraud"], yticklabels=["non-fraud", "fraud"])
    ax.set_ylim([0,2])
    plt.title('Confusion Matrix')
    plt.ylabel('Real Classes')
    plt.xlabel('Predicted Classes')
    plt.show()

In [ ]:
plot_confusion_matrix(y_test, smote_preds)

In [ ]:
from sklearn.metrics import classification_report

In [ ]:
print(classification_report(
    y_test, smote_preds, target_names=['non-fraud', 'fraud']))

Due to XGBoost's inherent randomness, your exact numbers may differ, but you'll probably notice a sharp rise in legitimate transactions being flagged as fraudulent (false positives). This occurs because SMOTE created so many synthetic fraud examples that they start bleeding into the same feature space as legitimate transactions.

Cohen's Kappa is more sensitive to false positives than balanced accuracy, so it drops considerably, along with precision and F1 scores for fraud cases. However, we can restore some balance by adjusting our decision threshold.

So far, we've been using 0.5 as our decision boundary for classifying transactions as fraudulent or legitimate. Let's experiment with different thresholds to see how they affect our results. We'll continue using balanced accuracy and Cohen's Kappa to measure performance.

In [ ]:
for thres in np.linspace(0.1, 0.9, num=9):
    smote_thres_preds = np.where(smote_raw_preds > thres, 1, 0)
    print("Threshold: {:.1f}".format(thres))
    print("Balanced accuracy = {:.3f}".format(balanced_accuracy_score(y_test, smote_thres_preds)))
    print("Cohen's Kappa = {:.3f}\n".format(cohen_kappa_score(y_test, smote_thres_preds)))

We can see that Cohen's Kappa keeps improving as we raise the threshold, without hurting balanced accuracy much. This gives us a nice tuning knob: keep the threshold low if you want to catch every possible fraud case, or bump it up if you're more worried about flagging legitimate transactions as fraud.

## Clean up

We'll keep the unsupervised and base XGBoost endpoints running so our Lambda function can handle incoming event streams. The solution will automatically clean up these endpoints when deleted, but make sure to delete the prediction endpoints when you're finished. You can do this from the Amazon SageMaker console under the Endpoints page, or just run `predictor_name.delete_endpoint()` right here.

In [ ]:
# Uncomment to clean up endpoints
dpl_smote_xgb_clf.delete_model()
dpl_smote_xgb_clf.delete_endpoint()
sagemaker_client = boto3.client('sagemaker', region_name=aws_region)
waiter = sagemaker_client.get_waiter('endpoint_deleted')
waiter.wait(EndpointName="{}-xgb-smote-endpoint".format(s3_prefix))

In [ ]:
# Uncomment to clean up all endpoints at once.
# dpl_xgb_clf.delete_model()
# dpl_xgb_clf.delete_endpoint()
# dpl_smote_xgb_clf.delete_model()
# dpl_smote_xgb_clf.delete_endpoint()
# sagemaker_client = boto3.client('sagemaker', region_name=aws_region)
# waiter = sagemaker_client.get_waiter('endpoint_deleted')
# waiter.wait(EndpointName="{}-xgb-smote-endpoint".format(s3_prefix))
# waiter.wait(EndpointName="{}-xgb-endpoint".format(s3_prefix))